In [1]:
!nvidia-smi  # confirm Tesla T4 + CUDA
!git clone https://github.com/iesxz-c/Final.git
%cd Final

Tue Sep 15 13:34:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
!pip install -q transformers pyyaml safetensors huggingface_hub
# ultralytics/cv2 not needed for this run; do NOT let pip downgrade torch

In [6]:
import os
os.environ['CCTV_ANOMALY_VIDEOS_DIR'] = '/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1'
os.environ['CCTV_NORMAL_VIDEOS_DIR'] = '/content/drive/MyDrive/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition'

In [8]:
!ls /content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1/Abuse/Abuse028_x264.mp4
!ls /content/drive/MyDrive/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition/Normal_Videos_246_x264.mp4


/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1/Abuse/Abuse028_x264.mp4
/content/drive/MyDrive/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition/Normal_Videos_246_x264.mp4


In [10]:
!mkdir -p data
!cp /content/drive/MyDrive/phase2c_155.json data/phase2c_155.json

In [11]:
!python -c "import json; d=json.load(open('data/phase2c_155.json')); print('videos:', len(d['videos'])); print('anomaly:', sum(v['ground_truth_category'] != 'Normal' for v in d['videos'])); print('normal:', sum(v['ground_truth_category'] == 'Normal' for v in d['videos']))"

videos: 155
anomaly: 112
normal: 43


In [13]:
!git pull
!git status
!git log -1 --oneline

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 6.76 KiB | 3.38 MiB/s, done.
From https://github.com/iesxz-c/Final
   a413b9b..fd88577  master     -> origin/master
Updating a413b9b..fd88577
Fast-forward
 src/pipeline/evaluate_quantitative.py | 348 ++++++++++++++++++++++++++++++++++
 tests/test_evaluate_quantitative.py   | 153 +++++++++++++++
 2 files changed, 501 insertions(+)
 create mode 100644 src/pipeline/evaluate_quantitative.py
 create mode 100644 tests/test_evaluate_quantitative.py
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean
fd88577 (HEAD -> master, origin/master, origin/HEAD) Phase 4B: quantitative evaluation


In [15]:
!python -m src.pipeline.extract_ucf_events \
  --input-manifest data/phase2c_155.json \
  --output-dir data/evidence/phase2c_ucf_large_smoke \
  --device cuda \
  --limit-videos 1

preprocessor_config.json: 100% 415/415 [00:00<00:00, 1.63MB/s]
config.json: 100% 1.45k/1.45k [00:00<00:00, 923kB/s]

model.safetensors: downloading bytes:  15% 187M/1.22G [00:01<00:04, 242MB/s, 13.5MB/s  ]
model.safetensors: downloading bytes:  24% 289M/1.22G [00:01<00:03, 244MB/s, 24.8MB/s  ]
model.safetensors: downloading bytes:  93% 1.13G/1.22G [00:04<00:00, 293MB/s, 88.7MB/s  ]
model.safetensors: reconstructing file:  44% 536M/1.22G [00:05<00:06, 107MB/s, 23.7MB/s  ]
model.safetensors: downloading bytes: 100% 1.16G/1.16G [00:09<00:00, 128MB/s, 90.6MB/s  ]
model.safetensors: reconstructing file: 100% 1.22G/1.22G [00:09<00:00, 134MB/s, 86.7MB/s  ]
Event model: OPear/videomae-large-finetuned-UCF-Crime (bc5f1c1058158a05f8710de2b7c7c372fe69062b) on cuda (16 frames @ 8.0fps, top-5)
[1/1] anomaly/Abuse/Abuse028_x264.mp4
  23 windows (video 30.00fps)

Wrote 1 videos, 23 surveillance events -> data/evidence/phase2c_ucf_large_smoke in 12.1s


In [16]:
!python -m src.pipeline.extract_ucf_events \
  --input-manifest data/phase2c_155.json \
  --output-dir data/evidence/phase2c_ucf_large \
  --device cuda

Event model: OPear/videomae-large-finetuned-UCF-Crime (bc5f1c1058158a05f8710de2b7c7c372fe69062b) on cuda (16 frames @ 8.0fps, top-5)
[1/155] anomaly/Abuse/Abuse028_x264.mp4
  23 windows (video 30.00fps)
[2/155] anomaly/Abuse/Abuse030_x264.mp4
  25 windows (video 30.00fps)
[3/155] anomaly/Arrest/Arrest001_x264.mp4
  38 windows (video 30.00fps)
[4/155] anomaly/Arrest/Arrest007_x264.mp4
  50 windows (video 30.00fps)
[5/155] anomaly/Arrest/Arrest024_x264.mp4
  57 windows (video 30.00fps)
[6/155] anomaly/Arrest/Arrest030_x264.mp4
  136 windows (video 30.00fps)
[7/155] anomaly/Arrest/Arrest039_x264.mp4
  248 windows (video 30.00fps)
[8/155] anomaly/Arson/Arson007_x264.mp4
  98 windows (video 30.00fps)
[9/155] anomaly/Arson/Arson009_x264.mp4
  12 windows (video 30.00fps)
[10/155] anomaly/Arson/Arson010_x264.mp4
  50 windows (video 30.00fps)
[11/155] anomaly/Arson/Arson011_x264.mp4
  20 windows (video 30.00fps)
[12/155] anomaly/Arson/Arson016_x264.mp4
  29 windows (video 30.00fps)
[13/155] ano

In [18]:
!python -m src.pipeline.extract_ucf_events \
  --input-manifest data/phase2c_remaining_702.json \
  --output-dir data/evidence/phase2c_ucf_remaining_smoke \
  --device cuda \
  --limit-videos 1

Event model: OPear/videomae-large-finetuned-UCF-Crime (bc5f1c1058158a05f8710de2b7c7c372fe69062b) on cuda (16 frames @ 8.0fps, top-5)
[1/1] anomaly/Abuse/Abuse001_x264.mp4
  43 windows (video 30.00fps)

Wrote 1 videos, 43 surveillance events -> data/evidence/phase2c_ucf_remaining_smoke in 20.2s


In [20]:
!git pull

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 5), reused 10 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 10.52 KiB | 2.63 MiB/s, done.
From https://github.com/iesxz-c/Final
   fd88577..ce91a30  master     -> origin/master
Updating fd88577..ce91a30
Fast-forward
 scripts/run_resumable_ucf.py          | 287 ++++++++++++++++++++++++++++++++++
 src/pipeline/evaluate_quantitative.py | 126 +++++++++++++++
 tests/test_evaluate_quantitative.py   |  79 ++++++++++
 tests/test_run_resumable.py           | 172 ++++++++++++++++++++
 4 files changed, 664 insertions(+)
 create mode 100644 scripts/run_resumable_ucf.py
 create mode 100644 tests/test_run_resumable.py


In [ ]:
!python scripts/run_resumable_ucf.py \
  --input-manifest data/phase2c_remaining_702.json \
  --output-dir data/evidence/phase2c_ucf_batches \
  --device cuda

TOTAL: 702
COMPLETED BEFORE START: 0
REMAINING: 702
CURRENT BATCH: batch_001 (50 videos)
